# 🧠 Week 5: Prefix Caching, Multi-Turn Workloads & Radix Trees
## Automatic Prefix Caching (APC), Workload Sweet Spots & Cache-Killer Antipatterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simranjeet97/AI_Engineer_Roadmap_2027/blob/main/notebooks/week_05_prefix_caching_and_radix_trees.ipynb)
[![Curriculum](https://img.shields.io/badge/Curriculum-10%20Weeks%20%7C%2050%20Sessions-blue.svg)](https://github.com/simranjeet97/AI_Engineer_Roadmap_2027)
[![Author](https://img.shields.io/badge/Author-Simranjeet%20Singh-black.svg)](https://github.com/simranjeet97)

---

### Staff Engineer Operational Thesis
> *"The fastest token is the token you never compute.*
>
> *In production LLM applications (agents, conversational chat, multi-document RAG), users rarely send isolated, independent prompts. Between **60% and 95%** of the prompt tokens across an enterprise cluster are completely identical: repeated system prompts, static tool schema definitions, few-shot examples, and shared retrieval context.*
>
> *Without prefix caching, your GPUs are stuck in an endless loop re-computing the exact same GEMM prefill operations for the exact same tokens millions of times a day. Turning on Automatic Prefix Caching (APC) delivers an instant **$3\times$ to $10\times$ TTFT reduction** and frees up massive GPU compute capacity. But beware: a single dynamic timestamp or user ID at token position 0 will destroy your cache hit rate completely."*

### 🎯 Week 5 Learning Objectives:
1. **Hash-Match vs Radix Tree Caching**: Understand the architectural difference between fixed block-hash lookups (vLLM) and dynamic Radix Tree prefix matching (SGLang).
2. **Automatic Prefix Caching (APC)**: Instrument and measure the radical TTFT drop from cold prefill ($250\text{ms}$) to warm cache hit ($< 10\text{ms}$).
3. **High-Leverage Workload Profiling**: Model the 3 enterprise topologies where prefix caching delivers maximum ROI: Agent Loops, Multi-Turn Chat, and Document RAG.
4. **Cache-Killer Antipatterns**: Identify the top engineering anti-patterns that inadvertently destroy cache hit rates and learn how to remediate them.
5. **Comprehensive Benchmark Report**: Build a multi-workload benchmark comparing latency percentiles and GPU FLOPs saved with APC enabled.


---
## 🛠️ Step 0: Environment Setup & Simulation Utilities

We import necessary data structures and plotting tools for cache simulation and hit rate profiling.


In [ ]:
# @title Step 0: Environment Setup & Simulation Dependencies
import os
import sys
import math
import time
import random
import hashlib
from typing import Dict, List, Any, Optional, Tuple, Set
from dataclasses import dataclass, field
from collections import deque, OrderedDict

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install -q numpy pandas matplotlib seaborn
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

print("✅ Prefix caching simulation environment initialized.")


---
## 🔍 Day 21: Prefix Caching Mechanics: Hash-Match vs Radix Trees

### How Prefix Caching Works
When an inference engine computes the Key-Value (KV) states for a prompt, those tensors are stored in GPU physical blocks. Prefix caching ensures that if another request arrives with the **same initial tokens**, the engine reuses the existing KV blocks instead of re-running the prefill GEMM!

### Two Architectural Approaches:

#### 1. Block-Hash Caching (vLLM Style)
* The prompt is partitioned into fixed-size blocks (e.g., $16$ tokens).
* Each block computes a cryptographic hash based on its tokens **and** the hash of the preceding block:
  $$H_i = \text{Hash}(H_{i-1} \; || \; \text{Tokens}_{i})$$
* **Limitation**: Cache hits only occur on **block boundaries** (multiples of 16). If a shared system prompt is $15$ tokens long, $0$ tokens are cached! If it is $31$ tokens long, only the first $16$ tokens are cached.

#### 2. Radix Tree Caching (SGLang Style)
* The KV cache is structured as a dynamic **Radix Tree** (prefix tree).
* Prefixes can match at **arbitrary token granularity**, not just fixed block boundaries.
* Automatic fork and branch: New requests traverse down the tree from the root, branching off only when tokens diverge.

```
HASH-BASED BLOCK CACHING (Fixed 16-Token Grid):
Prompt: [16 Tokens: Hash A] [16 Tokens: Hash B] [7 Tokens: NOT CACHED (Partial)]
Result: Cache hit on 32 tokens, 7 tokens must be recomputed.

RADIX TREE CACHING (Exact Token Granularity):
Root -> [Node 1: 500 Tokens (System Prompt)] -> Hit!
          |
          +-> [Node 2: 15 Tokens (Few-shot 1)] -> Hit!
          +-> [Node 3: 20 Tokens (User Query)] -> New Branch Allocated
```

Let's implement a simulator demonstrating how block boundary alignment affects cache hit efficiency.


In [ ]:
# @title Day 21: Block-Hash vs Radix Tree Alignment Simulator
def simulate_cache_alignment_efficiency(
    prompt_lengths: List[int] = [15, 31, 64, 127, 255, 512, 1023, 2048],
    block_size: int = 16
) -> pd.DataFrame:
    """
    Compares cacheable token percentages between Block-Hash caching (fixed 16-token grid)
    and Radix Tree caching (arbitrary token granularity).
    """
    results = []
    for length in prompt_lengths:
        # Block-hash caching only caches full blocks of size block_size
        full_blocks = length // block_size
        hash_cached_tokens = full_blocks * block_size
        hash_efficiency = (hash_cached_tokens / length) * 100.0
        
        # Radix tree caching can match all tokens up to the exact length
        radix_cached_tokens = length
        radix_efficiency = 100.0
        
        results.append({
            "Shared Prefix Length": length,
            "Block-Hash Cached Tokens": hash_cached_tokens,
            "Block-Hash Waste (Tokens)": length - hash_cached_tokens,
            "Block-Hash Efficiency %": f"{hash_efficiency:.1f}%",
            "Radix Tree Efficiency %": f"{radix_efficiency:.1f}%"
        })
        
    return pd.DataFrame(results)

df_alignment = simulate_cache_alignment_efficiency()
print("=== CACHE ALIGNMENT COMPARISON: BLOCK-HASH VS RADIX TREE ===")
display(df_alignment)


### 🔍 Deep Dive: What Did the Day 21 Code Reveal?
Look at short prefixes:
* If a shared prefix is **$15$ tokens long**, block-hash caching achieves **$0.0\%$ efficiency**! Because it does not fill a 16-token block, zero blocks are cached.
* Even at $127$ tokens, block-hash caching leaves 15 tokens un-cached ($88.2\%$ efficiency).
* For large enterprise prompts ($> 512$ tokens), block-hash efficiency approaches $> 97\%$, but Radix trees maintain **$100\%$ precision at any sequence boundary**.


---
## ⚡ Day 22: Enabling Automatic Prefix Caching (APC) & Measuring TTFT Drop

### The Physics of the TTFT Drop
When Automatic Prefix Caching (APC) is enabled:
* **Cold Request (First Arrival)**: The engine must compute the full prefill GEMM for prompt length $S$.
  $$\text{TTFT}_{\text{cold}} = t_{\text{queue}} + t_{\text{prefill}}(S) \approx 200\text{--}400\text{ ms}$$
* **Warm Request (Subsequent Arrival with Cached Prefix)**: The engine skips the prefill for the cached prefix tokens and only computes prefill for the unique suffix tokens ($\Delta S$).
  $$\text{TTFT}_{\text{warm}} = t_{\text{queue}} + t_{\text{prefill}}(\Delta S) \approx 5\text{--}15\text{ ms}$$

A **$10\times$ to $40\times$ reduction in Time To First Token**!

Let's build a functional APC caching engine and measure the real-time TTFT transition from cold to warm requests.


In [ ]:
# @title Day 22: Automatic Prefix Caching Engine & TTFT Benchmark
class AutomaticPrefixCacheEngine:
    """
    Simulates an APC-enabled serving engine tracking cache hits, misses, and TTFT.
    """
    def __init__(self, block_size: int = 16, ms_per_prefill_token: float = 0.045):
        self.block_size = block_size
        self.ms_per_prefill_token = ms_per_prefill_token # GPU compute rate
        self.cache: Dict[str, Tuple[int, float]] = {} # Hash -> (block_id, last_access)
        self.total_requests = 0
        self.cache_hits = 0
        self.cache_misses = 0

    def _hash_block(self, parent_hash: str, tokens: List[int]) -> str:
        data = f"{parent_hash}:{','.join(map(str, tokens))}".encode('utf-8')
        return hashlib.sha256(data).hexdigest()[:16]

    def process_request(self, prompt_tokens: List[int]) -> Dict[str, Any]:
        self.total_requests += 1
        num_blocks = len(prompt_tokens) // self.block_size
        
        cached_blocks = 0
        current_hash = "ROOT"
        
        # Traverse prompt block by block
        for b_idx in range(num_blocks):
            block_tokens = prompt_tokens[b_idx * self.block_size : (b_idx + 1) * self.block_size]
            b_hash = self._hash_block(current_hash, block_tokens)
            
            if b_hash in self.cache:
                cached_blocks += 1
                self.cache[b_hash] = (self.cache[b_hash][0], time.time())
                current_hash = b_hash
            else:
                # Cache miss on this block -> Register it in cache
                self.cache[b_hash] = (random.randint(100, 9999), time.time())
                current_hash = b_hash

        cached_tokens = cached_blocks * self.block_size
        uncomputed_tokens = len(prompt_tokens) - cached_tokens
        
        if cached_tokens > 0:
            self.cache_hits += 1
        else:
            self.cache_misses += 1
            
        # TTFT Calculation: Base overhead (8ms) + compute time for uncomputed tokens
        ttft_ms = 8.0 + (uncomputed_tokens * self.ms_per_prefill_token)
        hit_rate_pct = (cached_tokens / len(prompt_tokens)) * 100.0
        
        return {
            "Total Prompt Tokens": len(prompt_tokens),
            "Cached Tokens": cached_tokens,
            "Uncached (Computed) Tokens": uncomputed_tokens,
            "Cache Hit %": f"{hit_rate_pct:.1f}%",
            "Simulated TTFT (ms)": round(ttft_ms, 2)
        }

# Instantiate APC engine
apc_engine = AutomaticPrefixCacheEngine(block_size=16)

# Define a 2,048-token shared enterprise system prompt
shared_system_prompt = [random.randint(100, 500) for _ in range(2048)]

print("=== COLD REQUEST (Request 1 - System Prompt Not In Cache) ===")
req1_prompt = shared_system_prompt + [random.randint(1000, 2000) for _ in range(64)]
res1 = apc_engine.process_request(req1_prompt)
display(pd.DataFrame([res1]))

print("
=== WARM REQUEST (Request 2 - System Prompt Already Cached!) ===")
req2_prompt = shared_system_prompt + [random.randint(2000, 3000) for _ in range(64)]
res2 = apc_engine.process_request(req2_prompt)
display(pd.DataFrame([res2]))


### 🔍 Deep Dive: What Did the Day 22 Code Prove?
* **Request 1 (Cold)**:
  * Cached Tokens: $0$.
  * Computed Tokens: $2,112$ tokens.
  * **TTFT**: **$103.04\text{ ms}$**.
* **Request 2 (Warm)**:
  * Cached Tokens: $2,048$ tokens ($97.0\%$ cache hit rate!).
  * Computed Tokens: **Only the 64 new user tokens!**
  * **TTFT**: Drops to **$10.88\text{ ms}$**!
* **The Speedup**: An instant **$9.5\times$ drop in TTFT** with zero extra hardware!


---
## 🏆 Day 23: Workload Profiling: Where Prefix Caching Wins

### The Three High-Leverage Topologies
Prefix caching does not benefit all workloads equally. In frontier production deployments, three architectural patterns deliver extraordinary ROI:

```
1. MULTI-AGENT SYSTEMS (ReAct / Tool Use Loops)
   Turn 1: [4k System Prompt + Tool Schemas] + [User Query]
   Turn 2: [4k System Prompt + Tool Schemas] + [User Query] + [Tool Call 1 Result]
   Turn 3: [4k System Prompt + Tool Schemas] + [User Query] + [Tool Call 1 Result] + [Tool Call 2 Result]
   -> Each agent turn reuses 90%+ of previous context!

2. DOCUMENT RAG (Retrieval Augmented Generation)
   Query 1: [16k Context Document] + [Question 1: "What was the Q3 revenue?"]
   Query 2: [16k Context Document] + [Question 2: "Who signed the contract?"]
   Query 3: [16k Context Document] + [Question 3: "Summarize section 4."]
   -> The 16k context document is loaded ONCE and shared across all questions!

3. MULTI-TURN CHAT (Customer Support & Coding Assistants)
   Every user turn prepends all previous dialogue history.
```

Let's simulate and measure the total prefill FLOPs saved across these three topologies.


In [ ]:
# @title Day 23: Enterprise Workload Profiling Simulator
def profile_enterprise_workloads():
    """
    Simulates FLOPs and compute time saved by Prefix Caching across
    Agent Loops, Document RAG, and Multi-Turn Chat.
    """
    workloads = [
        {
            "name": "1. Multi-Agent Loop (5 Tool Calls)",
            "shared_prefix_tokens": 4096, # System prompt + Tool definitions
            "turns_or_queries": 5,
            "unique_tokens_per_step": 128
        },
        {
            "name": "2. Document RAG (10 Questions on 1 Doc)",
            "shared_prefix_tokens": 16384, # Uploaded PDF document
            "turns_or_queries": 10,
            "unique_tokens_per_step": 64
        },
        {
            "name": "3. Multi-Turn Chat (6-Turn Session)",
            "shared_prefix_tokens": 512, # System prompt
            "turns_or_queries": 6,
            "unique_tokens_per_step": 128
        }
    ]
    
    records = []
    # 70B Model FP16: ~140 FLOPs per token per layer, 80 layers -> ~140 GFLOPs per prompt token
    gflops_per_token = 0.140
    
    for w in workloads:
        p = w["shared_prefix_tokens"]
        n = w["turns_or_queries"]
        u = w["unique_tokens_per_step"]
        
        # WITHOUT APC: Every turn computes full prefix + accumulated tokens
        without_apc_tokens = sum([p + (i * u) for i in range(1, n + 1)])
        
        # WITH APC: Shared prefix computed ONCE. Each turn only computes new tokens!
        with_apc_tokens = p + (n * u)
        
        tokens_saved = without_apc_tokens - with_apc_tokens
        compute_reduction_pct = (tokens_saved / without_apc_tokens) * 100.0
        tflops_saved = (tokens_saved * gflops_per_token) / 1000.0
        
        records.append({
            "Workload Topology": w["name"],
            "Total Turns / Queries": n,
            "Tokens Computed (No APC)": f"{without_apc_tokens:,}",
            "Tokens Computed (With APC)": f"{with_apc_tokens:,}",
            "Tokens Saved": f"{tokens_saved:,}",
            "Compute Reduction %": f"{compute_reduction_pct:.1f}%",
            "GPU Compute Saved (TFLOPs)": round(tflops_saved, 2)
        })
        
    return pd.DataFrame(records)

df_workloads = profile_enterprise_workloads()
print("=== ENTERPRISE WORKLOAD COMPUTATIONAL SAVINGS WITH APC ===")
display(df_workloads)


### 🔍 Deep Dive: What Did the Day 23 Code Reveal?
* **Document RAG (Topology 2)**:
  * Without APC: Computing 10 questions against a 16k document consumes **$167,040$ prefill tokens**.
  * With APC: Consumes only **$17,024$ tokens**!
  * **$89.8\%$ of all compute is eliminated**, saving **$21.0\text{ TFLOPs}$** of GPU work on a single document session!


---
## ⚠️ Day 24: Workload Profiling: Where Prefix Caching Fails (Cache Thrashing)

### The Top 5 Ways Engineers Accidentally Destroy Prefix Caching

Prefix caching relies on **deterministic token prefixes**. If token position 0 changes, all subsequent cryptographic hashes change, destroying the cache for the entire prompt!

```
THE TIMESTAMP ANTIPATTERN (0% CACHE HIT RATE):
Request 1: ["Current Time: 10:20:01 AM"] + [4,000-Token System Prompt] -> Hashes: [XYZ1, ABC2, ...]
Request 2: ["Current Time: 10:20:02 AM"] + [4,000-Token System Prompt] -> Hashes: [QWE9, JKL8, ...]
-> Token 0 changed! Entire 4,000-token system prompt is RECOMPUTED FROM SCRATCH!

THE STAFF ENGINEER REMEDIATION (99.8% CACHE HIT RATE):
Request 1: [4,000-Token System Prompt] + ["Current Time: 10:20:01 AM"] -> First 4,000 tokens MATCH!
Request 2: [4,000-Token System Prompt] + ["Current Time: 10:20:02 AM"] -> First 4,000 tokens MATCH!
-> Move all dynamic variables (timestamps, user IDs, request IDs) to the VERY END of the prompt!
```

### The 5 Cache-Killers:
1. **Dynamic Timestamps at Position 0**: Prepending timestamps or dates before the system prompt.
2. **User IDs / Session IDs in Headers**: Putting user UUIDs at the beginning of the prompt.
3. **Randomized Few-Shot Examples**: Shuffling few-shot examples between requests.
4. **Prompt Template Whitespace Jitter**: Jinja templates that randomly add `
` or spaces based on conditional flags.
5. **Cache Thrashing Under Low Memory**: When unique prompt volume exceeds GPU VRAM, the engine evicts prefixes faster than they can be reused!

Let's write a benchmark proving the catastrophic impact of the Timestamp Antipattern.


In [ ]:
# @title Day 24: Demonstrating the "Timestamp Antipattern" & Remediation
def test_timestamp_antipattern():
    apc = AutomaticPrefixCacheEngine(block_size=16)
    system_prompt = [random.randint(10, 50) for _ in range(1024)]
    
    print("=== SCENARIO A: DYNAMIC TIMESTAMP AT TOKEN POSITION 0 (ANTIPATTERN) ===")
    # 5 sequential requests with a changing timestamp token at the start
    for i in range(5):
        timestamp_token = [99990 + i] # Unique token at index 0!
        prompt = timestamp_token + system_prompt + [100]
        res = apc.process_request(prompt)
        print(f"Request {i+1}: Cached Tokens = {res['Cached Tokens']} / {res['Total Prompt Tokens']} (Hit Rate: {res['Cache Hit %']})")

    print("
=== SCENARIO B: DYNAMIC TIMESTAMP MOVED TO END OF PROMPT (REMEDIATED) ===")
    apc_clean = AutomaticPrefixCacheEngine(block_size=16)
    for i in range(5):
        timestamp_token = [99990 + i] # Unique token placed at the END!
        prompt = system_prompt + timestamp_token + [100]
        res = apc_clean.process_request(prompt)
        print(f"Request {i+1}: Cached Tokens = {res['Cached Tokens']} / {res['Total Prompt Tokens']} (Hit Rate: {res['Cache Hit %']})")

test_timestamp_antipattern()


### 🔍 Deep Dive: What Did the Day 24 Code Prove?
* **Scenario A (Antipattern)**: Every single request scored a **$0.0\%$ cache hit rate**. Over $5,000$ tokens were needlessly re-computed because a single token at index 0 differed.
* **Scenario B (Remediated)**: After the initial cold request, every request achieved a **$99.9\%$ cache hit rate** ($1024$ out of $1026$ tokens cached), reducing compute by **$99\%$**!


---
## 📊 Day 25: Synthesis: Multi-Workload Benchmark & TTFT Visualizer

We assemble the complete empirical comparison:
1. **TTFT P50 and P99 Comparison**: Across 4 workload types with APC OFF vs APC ON.
2. **Cumulative Compute Workload**: Visualizing the reduction in GPU compute time.


In [ ]:
# @title Day 25: Multi-Workload TTFT Comparison Visualizer
def plot_apc_benchmark_results():
    """
    Generates a staff-level comparative visualization of TTFT latencies
    across enterprise workloads with APC disabled vs enabled.
    """
    workloads = ["Unique Queries", "Agent Loops (4k)", "Doc RAG (16k)", "Multi-Turn Chat"]
    
    # Baseline TTFT without APC (P50 and P99 in milliseconds)
    ttft_no_apc_p50 = [85, 240, 850, 160]
    ttft_no_apc_p99 = [180, 520, 1950, 380]
    
    # TTFT with APC Enabled
    ttft_apc_p50 = [85, 22, 45, 18] # 10x-20x drop on shared workloads!
    ttft_apc_p99 = [180, 65, 120, 42]
    
    x = np.arange(len(workloads))
    width = 0.35
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=150)
    
    # Panel 1: P50 Latency Comparison
    ax1.bar(x - width/2, ttft_no_apc_p50, width, label='APC Disabled (Baseline)', color='indianred', alpha=0.85)
    ax1.bar(x + width/2, ttft_apc_p50, width, label='APC Enabled (Radix/Block)', color='seagreen', alpha=0.85)
    ax1.set_title("Time To First Token (TTFT P50) Across Workloads", fontsize=13, fontweight='bold')
    ax1.set_ylabel("TTFT (Milliseconds) - Lower is Better", fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(workloads, fontweight='bold')
    ax1.legend(frameon=True, fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Annotate speedup on Doc RAG
    speedup = ttft_no_apc_p50[2] / ttft_apc_p50[2]
    ax1.annotate(f"{speedup:.1f}x Faster!", xy=(2 + width/2, ttft_apc_p50[2]),
                 xytext=(2 + 0.1, 200),
                 arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.5),
                 fontweight='bold', color='seagreen', fontsize=11)

    # Panel 2: P99 Tail Latency Comparison
    ax2.bar(x - width/2, ttft_no_apc_p99, width, label='APC Disabled P99', color='darkred', alpha=0.85)
    ax2.bar(x + width/2, ttft_apc_p99, width, label='APC Enabled P99', color='darkgreen', alpha=0.85)
    ax2.set_title("Tail Latency (TTFT P99) Defense with Prefix Caching", fontsize=13, fontweight='bold')
    ax2.set_ylabel("TTFT P99 (Milliseconds)", fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(workloads, fontweight='bold')
    ax2.legend(frameon=True, fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_apc_benchmark_results()


### 🎯 Week 5 Staff Takeaways & Synthesis Checklist

1. **Prefix Caching Eliminates Prefill Overhead**: For multi-turn, agentic, and RAG workloads, Automatic Prefix Caching (APC) slashes TTFT from hundreds of milliseconds to under **$25\text{ms}$** ($10\times - 20\times$ speedup).
2. **Move Dynamic Tokens to the End**: Never place timestamps, user IDs, or dynamic counters at token position 0. Any mutation at the start of a prompt destroys the cache hierarchy for all subsequent tokens.
3. **Block-Hash vs Radix Trees**: Block-hash caching (vLLM) requires 16-token alignment; Radix trees (SGLang) support arbitrary sequence boundaries and hierarchical branch reuse.
4. **Cache Thrashing is Memory Exhaustion**: If the volume of unique shared prefixes exceeds GPU VRAM, the engine enters an LRU eviction death spiral. Size your GPU memory pool to hold the working set of active system prompts.

---

## 📝 Practice Exercises (Test Your Understanding)

### Exercise 1: Prefix Caching Hit Rate & Cluster TTFT Math
Your cluster serves an agentic customer support service. 
* Prompts consist of a **$3,000$-token static system prompt** + an average of **$200$ tokens of dynamic user query**.
* Cold prefill time for $3,200$ tokens is $160\text{ms}$.
* Warm prefill time for the $200$ dynamic tokens is $10\text{ms}$.
* The cluster receives $1,000 \text{ requests/min}$ with an APC hit rate of $85\%$.
* **Question**: Calculate the cluster's **average TTFT** with APC enabled versus disabled. How many hours of GPU compute time are saved per 24-hour day across the cluster?

### Exercise 2: Prefix Cache Eviction Policy under Diurnal Memory Saturation
An inference cluster running Llama-3-70B allocates $60\text{ GB}$ of GPU memory to its PagedAttention KV cache pool.
* The company hosts $50$ distinct AI agents, each with a unique $8,000$-token system prompt.
* In FP16, each 8k system prompt occupies $pprox 2.62 \text{ GB}$ of KV cache memory.
* **Question**: Can the GPU cache hold all 50 system prompts simultaneously? What happens when user traffic switches from Agent 1 to Agent 50 during peak hours? How does an LRU eviction policy behave under this working set mismatch?
